In [2]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
import sqlite3


llm = ChatAnthropic(
    model_name='claude-opus-4-7',
        base_url='https://api.agentshop247.com/api/claude',
        temperature=0,
        timeout=None,
        stop=None,
)

def message_limit(old: list[BaseMessage], new: list[BaseMessage]):
    merge = old + new
    return merge[-5:]

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def receive_input(state: AgentState):
    prompt_input = input("User: ")
    return {
        'messages': [HumanMessage(content = prompt_input)]
    }
def process(state: AgentState) -> dict:
    full_response = llm.invoke(state['messages'])
    for block in full_response.content:
        if isinstance(block, dict) and block['type'] == 'text':
            response = block['text']
            print(f"Agent: {response}")
            break
    return {
        'messages': [full_response]
    }

def router(state: AgentState):
    if state['messages'][-1].content == 'exit':
        return "end"
    else: 
        return "llm"

graph = StateGraph(AgentState)
graph.add_node('agent', process)
graph.add_node('human', receive_input)
graph.add_edge(START, 'human')
graph.add_edge('agent', END)
graph.add_edge('agent', 'human')
graph.add_conditional_edges(
    'human', 
    router,
    {
        'end': END,
        'llm': 'agent'
    }
)


conn = sqlite3.connect("test.sqlite", check_same_thread = False)
memory = SqliteSaver(conn)

app = graph.compile(checkpointer=memory)

config = {"configurable":{"thread_id":"1"}}

res = app.invoke({"messages": []},config=config)


Agent: Hello! It looks like you might be introducing yourself. I'm Claude. How can I help you today?
Agent: Nice, you're at an interesting age — lots of possibilities ahead. Is there something I can help you with today, or just wanted to chat?
Agent: Got it, Nam. Thanks for sharing a bit about yourself. If you ever need help with coding, debugging, or thinking through a project, I'm here. What brings you to Claude today?


In [3]:
history = list(app.get_state_history(config = config))
for block in history:    # Block is StateSnapshot
    message = block.values['messages']
    for i in message:
        if isinstance(i, HumanMessage):
            print("Human: ", i.content, end = " ")
            print("|",end = " ")
        elif isinstance(i, AIMessage):
            print("AI: ", end = " ")
            for j in i.content:
                if isinstance(j, dict) and j['type'] == 'text':
                    print(j['text'], end = " ")
                    break
            print("|", end = " ")
    print(block.config['configurable']['checkpoint_id'], block.next)

Human:  Im Nam | AI:  Hello! It looks like you might be introducing yourself. I'm Claude. How can I help you today? | Human:  20 years old | AI:  Nice, you're at an interesting age — lots of possibilities ahead. Is there something I can help you with today, or just wanted to chat? | Human:  Male | AI:  Got it, Nam. Thanks for sharing a bit about yourself. If you ever need help with coding, debugging, or thinking through a project, I'm here. What brings you to Claude today? | Human:  exit | 1f19c83e-b841-6602-8007-802a1df139b4 ()
Human:  Im Nam | AI:  Hello! It looks like you might be introducing yourself. I'm Claude. How can I help you today? | Human:  20 years old | AI:  Nice, you're at an interesting age — lots of possibilities ahead. Is there something I can help you with today, or just wanted to chat? | Human:  Male | AI:  Got it, Nam. Thanks for sharing a bit about yourself. If you ever need help with coding, debugging, or thinking through a project, I'm here. What brings you to C

In [ ]:
config_rewind = {
                "configurable": {
                    "thread_id": "1",
                    "checkpoint_id": "1f19c83d-a519-6904-8004-287a408859fb"
                    }
}   
app.invoke(None,config= config_rewind)

Agent: Honestly, no — I can't tell from your name alone. "Im Nam" is a Korean name, and Korean names can be used by people of any gender without much distinction.

If you'd like to share, I'm happy to know, but it's not necessary. How can I help you?
Agent: Got it, thanks for sharing! So, Im Nam — is there anything I can help you with today? Whether it's a question, a project, or just curiosity about something, I'm here.


{'messages': [HumanMessage(content='Im Nam', additional_kwargs={}, response_metadata={}, id='3f45d3af-872d-4803-a365-f6957bc1d581'),
  AIMessage(content=[{'signature': '7951ecc929f3b02cc22f252742b659bef1cc8da3c37745341e0e89e486178dbc', 'thinking': 'The user said "Im Nam" - this appears to be a greeting or introduction, possibly in Korean or a name. "Im Nam" could be:\n\n1. A Korean name (surname "Im" or "Lee" with given name "Nam")\n2. A casual greeting or statement\n3. An incomplete message\n\nWithout more context, I\'ll respond warmly and ask how I can help.', 'type': 'thinking'}, {'text': "Hello! It looks like you might be introducing yourself. I'm Claude. How can I help you today?", 'type': 'text'}], additional_kwargs={}, response_metadata={'id': '06d60d0ffd002c21e4dc71e91c6d82dd', 'container': None, 'model': 'claude-opus-4-7', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': None, 'cache_creation_input_tokens': None, 'cache_read_i

In [5]:
history = list(app.get_state_history(config = config))
for block in history:    # Block is StateSnapshot
    message = block.values['messages']
    for i in message:
        if isinstance(i, HumanMessage):
            print("Human: ", i.content, end = " ")
            print("|",end = " ")
        elif isinstance(i, AIMessage):
            print("AI: ", end = " ")
            for j in i.content:
                if isinstance(j, dict) and j['type'] == 'text':
                    print(j['text'], end = " ")
                    break
            print("|", end = " ")
    print(block.config['configurable']['checkpoint_id'], block.next)

Human:  Im Nam | AI:  Hello! It looks like you might be introducing yourself. I'm Claude. How can I help you today? | Human:  20 years old | AI:  Nice, you're at an interesting age — lots of possibilities ahead. Is there something I can help you with today, or just wanted to chat? | Human:  you know im male or female ? | AI:  Honestly, no — I can't tell from your name alone. "Im Nam" is a Korean name, and Korean names can be used by people of any gender without much distinction.

If you'd like to share, I'm happy to know, but it's not necessary. How can I help you? | Human:  female | AI:  Got it, thanks for sharing! So, Im Nam — is there anything I can help you with today? Whether it's a question, a project, or just curiosity about something, I'm here. | Human:  exit | 1f19c844-5ae2-6ca8-800a-8e42ee13c947 ()
Human:  Im Nam | AI:  Hello! It looks like you might be introducing yourself. I'm Claude. How can I help you today? | Human:  20 years old | AI:  Nice, you're at an interesting age

Đã quên Male

In [6]:
res = app.invoke({'messages': []},config=config)

Agent: **Tóm tắt thông tin về bạn:**

- **Tên:** Im Nam
- **Tuổi:** 20
- **Giới tính:** Nữ

Bạn cần tôi giúp gì thêm không? 😊


Quay lại Male

In [7]:
new_config = {"configurable":{"thread_id": "1","checkpoint_id":"1f19c83e-2695-628a-8006-442256b0b4fb"}}

app.invoke({'messages': []}, config = new_config)

Agent: Dưới đây là thông tin bạn đã chia sẻ:

| Thông tin | Chi tiết |
|-----------|----------|
| **Tên** | Nam |
| **Tuổi** | 20 |
| **Giới tính | Nam |

Bạn cần tôi giúp gì thêm không?


{'messages': [HumanMessage(content='Im Nam', additional_kwargs={}, response_metadata={}, id='3f45d3af-872d-4803-a365-f6957bc1d581'),
  AIMessage(content=[{'signature': '7951ecc929f3b02cc22f252742b659bef1cc8da3c37745341e0e89e486178dbc', 'thinking': 'The user said "Im Nam" - this appears to be a greeting or introduction, possibly in Korean or a name. "Im Nam" could be:\n\n1. A Korean name (surname "Im" or "Lee" with given name "Nam")\n2. A casual greeting or statement\n3. An incomplete message\n\nWithout more context, I\'ll respond warmly and ask how I can help.', 'type': 'thinking'}, {'text': "Hello! It looks like you might be introducing yourself. I'm Claude. How can I help you today?", 'type': 'text'}], additional_kwargs={}, response_metadata={'id': '06d60d0ffd002c21e4dc71e91c6d82dd', 'container': None, 'model': 'claude-opus-4-7', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': None, 'cache_creation_input_tokens': None, 'cache_read_i

In [8]:
history = list(app.get_state_history(config = config))
for block in history:    # Block is StateSnapshot
    message = block.values['messages']
    for i in message:
        if isinstance(i, HumanMessage):
            print("Human: ", i.content, end = " ")
            print("|",end = " ")
        elif isinstance(i, AIMessage):
            print("AI: ", end = " ")
            for j in i.content:
                if isinstance(j, dict) and j['type'] == 'text':
                    print(j['text'], end = " ")
                    break
            print("|", end = " ")
    print(block.config['configurable']['checkpoint_id'], block.next)

Human:  Im Nam | AI:  Hello! It looks like you might be introducing yourself. I'm Claude. How can I help you today? | Human:  20 years old | AI:  Nice, you're at an interesting age — lots of possibilities ahead. Is there something I can help you with today, or just wanted to chat? | Human:  Male | AI:  Got it, Nam. Thanks for sharing a bit about yourself. If you ever need help with coding, debugging, or thinking through a project, I'm here. What brings you to Claude today? | Human:  tổng hợp lại thông tin của tôi | AI:  Dưới đây là thông tin bạn đã chia sẻ:

| Thông tin | Chi tiết |
|-----------|----------|
| **Tên** | Nam |
| **Tuổi** | 20 |
| **Giới tính | Nam |

Bạn cần tôi giúp gì thêm không? | Human:  exit | 1f19c84e-3be1-6016-800b-e5b7168199a2 ()
Human:  Im Nam | AI:  Hello! It looks like you might be introducing yourself. I'm Claude. How can I help you today? | Human:  20 years old | AI:  Nice, you're at an interesting age — lots of possibilities ahead. Is there something I can 